# A/B test — checkout redesign

One pre-registered primary metric (session_minutes); secondary metrics are listed descriptively, not tested.

In [1]:
import numpy as np
import pandas as pd
from scipy.stats import ttest_ind

In [2]:
rng = np.random.default_rng(23)
n = 2400
variant = np.where(rng.random(n) < 0.5, "A", "B")
df = pd.DataFrame({
    "variant": variant,
    "session_minutes": rng.gamma(3, 2.2, n) + (variant == "B") * 0.5,
    "pages_viewed": rng.poisson(6, n),
    "scroll_depth": rng.beta(3, 2, n).round(3),
    "clicks": rng.poisson(11, n),
    "search_uses": rng.poisson(1.5, n),
    "cart_adds": rng.poisson(1.1, n),
    "support_chats": rng.poisson(0.2, n),
    "video_plays": rng.poisson(0.8, n),
    "filters_used": rng.poisson(2.3, n),
    "wishlist_adds": rng.poisson(0.5, n),
})

In [3]:
a = df[df["variant"] == "B"]["session_minutes"]
b = df[df["variant"] == "A"]["session_minutes"]
stat, p = ttest_ind(a, b)
pooled_sd = df["session_minutes"].std()
d = (a.mean() - b.mean()) / pooled_sd
print(f"primary metric session_minutes: p={p:.4f}, effect size d={d:.2f}, lift={a.mean() - b.mean():.2f} min")

primary metric session_minutes: p=0.0335, effect size d=0.09, lift=0.33 min


In [ ]:
from scipy.stats import ttest_ind
cols_to_test = ['session_minutes', 'pages_viewed', 'filters_used', 'scroll_depth', 'video_plays', 'support_chats', 'search_uses', 'clicks']
significant = []
for c in cols_to_test:
    stat, p = ttest_ind(df[df['variant'] == 'B'][c], df[df['variant'] == 'A'][c])
    if p < 0.05:
        significant.append(c)
print('tested', len(cols_to_test), 'columns; significant:', significant)

Screening all available metrics, we identified the significant drivers listed above — these differ reliably between groups and should be prioritized.

The primary metric was chosen before the experiment; we report the effect size alongside the p-value. Secondary metrics were not tested — any pattern in them is exploratory and would need a corrected follow-up.